# 01 — Explore the B11 catalog

Per `plan.md` §10 / §11 step 1. Loads `Data/J_AJ_141_23/table7.dat`,
renders diameter / Vexp / axial-ratio histograms, and breaks the 1046
holes down by hole type (1/2/3) per galaxy. Use this to confirm §2.4
decisions about the type-1 split before locking the LOGO run.

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hishells.catalog import load_catalog, LOGO_GALAXIES_19

In [ ]:
REPO = Path('..').resolve()
cat = load_catalog(REPO / 'Data' / 'J_AJ_141_23')
holes, gals = cat.holes, cat.galaxies
print(f'{len(holes)} holes across {holes["galaxy_id"].nunique()} galaxies')
print(f'{len(LOGO_GALAXIES_19)} galaxies reachable for LOGO ({set(holes["galaxy_id"]) - set(LOGO_GALAXIES_19)} excluded)')
holes.head()

## Hole-type breakdown

Type 1 = stalled (no measurable Vexp), 2 = one-sided expansion,
3 = two-sided expansion (textbook ellipse). Per §2.4 the v1 default
trains on types {2,3} only — use the bar chart below to confirm that
still leaves enough positives per galaxy.

In [ ]:
by_type = holes.groupby(['galaxy_id', 'hole_type']).size().unstack(fill_value=0)
by_type = by_type.reindex(LOGO_GALAXIES_19)
ax = by_type.plot.barh(stacked=True, figsize=(8, 6))
ax.set_xlabel('number of holes')
ax.set_ylabel('galaxy')
ax.set_title('B11 holes per galaxy, stacked by hole type (v1 trains on 2+3)')
ax.legend(title='type')
plt.tight_layout()

In [ ]:
summary = by_type.copy()
summary['total'] = summary.sum(axis=1)
summary['types_2_3'] = summary.get(2, 0) + summary.get(3, 0)
summary['frac_2_3'] = summary['types_2_3'] / summary['total']
summary.style.format({'frac_2_3': '{:.0%}'})

## Diameter / Vexp / axial-ratio histograms

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (col, label, bins) in zip(
    axes,
    [
        ('diameter_pc', 'diameter [pc]', np.logspace(np.log10(80), np.log10(2200), 30)),
        ('vexp_kms', 'V_exp [km/s] (types 2+3 only)', np.linspace(0, 40, 25)),
        ('axial_ratio', 'axial ratio b/a', np.linspace(0.0, 1.0, 25)),
    ],
):
    sub = holes if col != 'vexp_kms' else holes[holes['hole_type'].isin([2, 3])]
    ax.hist(sub[col].dropna(), bins=bins, color='tab:blue', edgecolor='k', alpha=0.8)
    ax.set_xlabel(label)
    if col == 'diameter_pc':
        ax.set_xscale('log')
axes[0].set_ylabel('count')
fig.tight_layout()

## Galactocentric radius vs hole diameter

B11 §5 reports that 23% of holes lie outside R25 and that shear bounds
hole age (and therefore hole size) in spirals. Use this scatter to
sanity-check the catalog and to bookmark large-radius outliers for
§06_failure_analysis.ipynb.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for ht, marker, label in [(1, 'o', 'type 1 (stalled)'), (2, 's', 'type 2'), (3, '^', 'type 3')]:
    sub = holes[holes['hole_type'] == ht]
    ax.scatter(sub['gc_radius_kpc'], sub['diameter_pc'], s=14, marker=marker, alpha=0.45, label=label)
ax.set_xlabel('R_GC [kpc]')
ax.set_ylabel('hole diameter [pc]')
ax.set_yscale('log')
ax.legend(loc='best')
fig.tight_layout()

## What this notebook locks

- Confirm that types {2,3} together leave ≥ ~20 holes per galaxy on at
  least 15/19 LOGO folds. If not, revisit §2.4.
- Confirm that diameter range matches `100 pc` to `~2 kpc`. If a galaxy
  has holes outside that range, double-check `Data/J_AJ_141_23/table7.dat`
  alignment in `tests/test_catalog.py`.